> **历史课程入口**：本 Notebook 为外部 PaddleYOLO 流程，仅供对照。当前中英文课程统一使用 [YOLO26n 实验](01-yolo26-rock-paper-scissors.ipynb)和正式 Rock-Paper-Scissors v1 数据；不要将这里的旧命令作为当前默认训练步骤。

# 目标检测模型构建流程

**基本的计算机视觉任务研发全流程** 


<br></br>
<center><img src="../assets/01-yolo11-rock-paper-scissors-1.png" width = "700"></center>
<center><br>图1：基本的视觉任务研发全流程</br></center>
<br></br>

**通用的视觉任务研发全流程**

目标检测模型构建流程如 **图2** 所示，和基本的计算机视觉任务研发全流程(**图1**)相比增加了**检测头结构**和**多尺度检测结构**2部分。目标检测既需要识别出物体的类别，还需要标示出每个目标的位置，因此我们增加检测头结构，用于计算预测框是否包含物体的概率、预测框位置坐标以及物体属于每个类别的概率。同时在目标检测中，检测的物体形状尺寸可能变化比较大，使用单一尺寸的特征图做预测容易造成漏检，而且像素点包含的语义信息可能不够丰富，难以提取到有效的特征模式，因此在目标检测中将高级层的特征图尺寸放大之后跟低层级的特征图进行融合，从而检测多尺度目标。


<br></br>
<center><img src="../assets/01-yolo11-rock-paper-scissors-2.png" width = "700"></center>
<center><br>图2：通用的视觉任务研发全流程</br></center>
<br></br>

* 数据处理：根据网络接收的数据格式，完成相应的预处理操作，保证模型正常读取，同时产生候选区域；
* 模型构建：设计目标检测网络结构；
* 特征提取：使用卷积神经网络提取特征；
* 检测头设计：计算预测框是否包含物体的概率、预测框位置坐标、物体属于每个类别概率；
* 多尺度检测：将高级层的特征图尺寸放大之后跟低层级的特征图进行融合；
* 损失函数：模型优化的目标；
* 模型评估：在模型训练中或训练结束后岁模型进行评估测试，观察准确率；
* 模型预测：使用训练好的模型进行测试，也需要准备数据和模型特征提取，最后对结果进行解析；
* 模型部署：通常需要将训练好的模型在特定环境(服务器、手机等)中运行，即在设备端运行推理任务。

接下来具体介绍流程中各节点的原理和代码实现。

# 框架选择

选择目标检测框架时，Ultralytics YOLO系列和百度的PaddlePaddle（特别是其PaddleDetection套件）都是非常优秀的选择。

| 特性维度       | Ultralytics YOLO                                             | PaddlePaddle (PaddleDetection)                               |
| -------------- | ------------------------------------------------------------ | ------------------------------------------------------------ |
| **核心定位**   | **用户友好、快速上手**的YOLO系列官方实现和框架               | **产业级、全流程**的深度学习平台和检测套件                   |
| **模型与算法** | 专注并**集成YOLO系列**（v3, v5, v8, v9, v10, v11等），更新迅速 | **算法丰富**，覆盖Faster R-CNN、YOLO、SSD等主流模型，并提供优化版本 |
| **易用性**     | **极高**。提供简洁的Python API和CLI命令，几行代码即可完成训练、推理、部署 | **中等**。依赖配置文件驱动，灵活性强，但学习曲线相对陡峭     |
| **部署支持**   | **优秀**。原生支持导出ONNX、TensorRT等格式，部署工具链成熟   | **强大**。提供**一键式压缩（剪裁、量化）和部署**能力，对国产硬件支持更好 |
| **社区与文档** | 全球社区活跃，英文文档为主。中文社区资料丰富但多为第三方翻译 | 中文文档和社区支持**极具优势**，本土化服务好                 |

**PaddleYOLO项目拉取，安装依赖**

Clone 代码库和安装 requirements.txt，环境需要在一个 Python>=3.7.0 下的环境，且需要安装 PaddlePaddle>=2.4.2。

```
git clone https://github.com/PaddlePaddle/PaddleYOLO  
cd PaddleYOLO
pip install -r requirements.txt  
pip install paddlepaddle-gpu==2.6.2  
```

简单推理，以yolov11为例，
参考：PaddleYOLO/tree/develop/configs/yolo11

CPU版本

```
# 推理单张图片
(env_paddle) root@776de05bea4b:/workspace/PaddleYOLO# CUDA_VISIBLE_DEVICES=-1 python tools/infer.py -c configs/yolo11/yolo11_n_600e_coco.yml -o use_gpu=False weights=https://paddledet.bj.bcebos.com/models/yolo11_n_600e_coco.pdparams --infer_img=demo/000000014439_640x640.jpg
[09/12 01:40:10] ppdet.utils.download INFO: Downloading yolo11_n_600e_coco.pdparams from https://paddledet.bj.bcebos.com/models/yolo11_n_600e_coco.pdparams
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10359/10359 [00:03<00:00, 3275.78KB/s]
[09/12 01:40:15] ppdet.utils.checkpoint INFO: ['model.model.23.dfl.conv.weight'] in pretrained weight is not used in the model, and its will not be loaded
[09/12 01:40:15] ppdet.utils.checkpoint INFO: Finish loading model weights: /root/.cache/paddle/weights/yolo11_n_600e_coco.pdparams
[09/12 01:40:15] ppdet.data.source.category WARNING: anno_file 'dataset/coco/annotations/instances_val2017.json' is None or not set or not exist, please recheck TrainDataset/EvalDataset/TestDataset.anno_path, otherwise the default categories will be used by metric_type.
[09/12 01:40:15] ppdet.data.source.category WARNING: metric_type: COCO, load default categories of COCO.
[09/12 01:40:15] ppdet.engine INFO: Test loader length is 1, test batch_size is 1.
[09/12 01:40:15] ppdet.engine INFO: Starting predicting ......

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.82it/s]
[09/12 01:40:15] ppdet.engine INFO: Detection bbox results save in output/000000014439_640x640.jpg
```

GPU版本

```
# 推理单张图片
CUDA_VISIBLE_DEVICES=0 python tools/infer.py -c configs/yolo11/yolo11_n_600e_coco.yml -o weights=https://paddledet.bj.bcebos.com/models/yolo11_n_600e_coco.pdparams --infer_img=demo/000000014439_640x640.jpg

# 推理文件中的所有图片
CUDA_VISIBLE_DEVICES=0 python tools/infer.py -c configs/yolo11/yolo11_n_600e_coco.yml -o weights=https://paddledet.bj.bcebos.com/models/yolo11_n_600e_coco.pdparams --infer_dir=demo
```

```
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO# CUDA_VISIBLE_DEVICES=0 python tools/infer.py -c configs/yolo11/yolo11_n_600e_coco.yml -o weights=https://paddledet.bj.bcebos.com/models/yolo11_n_600e_coco.pdparams --infer_img=demo/000000014439_640x640.jpg
Warning: import ppdet from source directory without installing, run 'python setup.py install' to install ppdet firstly
W0912 05:38:13.989795  1674 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.2, Runtime API Version: 11.8
W0912 05:38:13.990681  1674 gpu_resources.cc:164] device: 0, cuDNN Version: 8.9.
[09/12 05:38:14] ppdet.utils.download INFO: Downloading yolo11_n_600e_coco.pdparams from https://paddledet.bj.bcebos.com/models/yolo11_n_600e_coco.pdparams
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10359/10359 [00:03<00:00, 3101.75KB/s]
[09/12 05:38:19] ppdet.utils.checkpoint INFO: ['model.model.23.dfl.conv.weight'] in pretrained weight is not used in the model, and its will not be loaded
[09/12 05:38:19] ppdet.utils.checkpoint INFO: Finish loading model weights: /root/.cache/paddle/weights/yolo11_n_600e_coco.pdparams
[09/12 05:38:19] ppdet.data.source.category WARNING: anno_file 'dataset/coco/annotations/instances_val2017.json' is None or not set or not exist, please recheck TrainDataset/EvalDataset/TestDataset.anno_path, otherwise the default categories will be used by metric_type.
[09/12 05:38:19] ppdet.data.source.category WARNING: metric_type: COCO, load default categories of COCO.
[09/12 05:38:19] ppdet.engine INFO: Test loader length is 1, test batch_size is 1.
[09/12 05:38:19] ppdet.engine INFO: Starting predicting ......

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.31it/s]
[09/12 05:38:20] ppdet.engine INFO: Detection bbox results save in output/000000014439_640x640.jpg
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO# 
```

更多操作参考： $PaddleYOLO/configs/yolo11/README.md$

# 预期效果

训练一个模型，能识别出手势： "石头（Rock）"，"布（Paper）"，"剪刀（Scissors）"

训练前： 
![image.png](../assets/01-yolo11-rock-paper-scissors-3.png)


训练后： 
![image.png](../assets/01-yolo11-rock-paper-scissors-4.png)



# 自定义的数据集

## 数据集介绍

目标检测的数据比分类复杂，一张图像中，需要标记出各个目标区域的位置和类别。

一般的目标区域位置用一个矩形框来表示，一般用以下3种方式表达：

|         表达方式    |                 说明               |
| :----------------: | :--------------------------------: |
|     x1,y1,x2,y2    | (x1,y1)为左上角坐标，(x2,y2)为右下角坐标  |  
|     x1,y1,w,h      | (x1,y1)为左上角坐标，w为目标区域宽度，h为目标区域高度  |
|     xc,yc,w,h    | (xc,yc)为目标区域中心坐标，w为目标区域宽度，h为目标区域高度  |  

常见的目标检测数据集如Pascal VOC采用的`[x1,y1,x2,y2]` 表示物体的bounding box, [COCO](https://cocodataset.org/#format-data)采用的`[x1,y1,w,h]` 表示物体的bounding box.

## 数据标注制作（略）

参考： https://bgithub.xyz/PaddlePaddle/PaddleYOLO/blob/develop/docs/tutorials/data/DetAnnoTools.md


## 数据集获取  
这里就不手动标注了，已下载好数据集在/workspace/DataSet/yolo目录下， road-sign-detection.zip 文件  
拷贝数据集到 /workspace/PaddleYOLO/dataset 目录下  


In [9]:
# !cp /DataSet/yolo/Rock-Paper-Scissors.zip /workspace/PaddleYOLO/dataset

In [10]:
# !unzip /workspace/PaddleYOLO/dataset/Rock-Paper-Scissors.zip -d /workspace/PaddleYOLO/dataset/

.     
├── Rock-Paper-Scissors    
│   ├── annotations    
│   │   ├── 0001_png.rf.5e009a9ec5fbf7a8e856ba7f413996af.xml  
│   │   ├── 0001_png.rf.5e22fcd367ffc26ff60b1c51ad0f660c.xml    
│   │   ├── 0001_png.rf.b7e30e943e30c208a0a0bcf446e6cce4.xml   
|   `-- images
|       |-- 0001_png.rf.5e009a9ec5fbf7a8e856ba7f413996af.jpg     
|       |-- 0001_png.rf.5e22fcd367ffc26ff60b1c51ad0f660c.jpg  
|       |-- 0001_png.rf.b7e30e943e30c208a0a0bcf446e6cce4.jpg    

查看文件个数是否匹配

尝试使用预训练模型进行推理，并不能识别出剪刀，石头，布


```shell
CUDA_VISIBLE_DEVICES=0 python tools/infer.py -c configs/yolo11/yolo11_s_600e_coco.yml -o weights=https://paddledet.bj.bcebos.com/models/yolo11_s_600e_coco.pdparams --infer_img=/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors/images/Photo-on-2-16-22-at-10-19-AM-9_jpg.rf.79e2cf3a561a19b33a74e9ea9477df76.jpg
```

![image.png](../assets/01-yolo11-rock-paper-scissors-3.png)

## 数据集制作-VOC

将数据划分为训练集和测试集
```
# 生成 label_list.txt 文件
>>echo -e "speedlimit\ncrosswalk\ntrafficlight\nstop" > label_list.txt

# 生成 train.txt、valid.txt和test.txt列表文件
>>ls images/*.png | shuf > all_image_list.txt
>>awk -F"/" '{print $2}' all_image_list.txt | awk -F".png" '{print $1}'  | awk -F"\t" '{print "images/"$1".png annotations/"$1".xml"}' > all_list.txt

# 训练集、验证集、测试集比例分别约80%、10%、10%。
>>head -n 88 all_list.txt > test.txt
>>head -n 176 all_list.txt | tail -n 88 > valid.txt
>>tail -n 701 all_list.txt > train.txt

# 删除不用文件
>>rm -rf all_image_list.txt all_list.txt

最终数据集文件组织结构为：

├── annotations
│   ├── road0.xml
│   ├── road1.xml
│   ├── road10.xml
│   |   ...
├── images
│   ├── road0.jpg
│   ├── road1.jpg
│   ├── road2.jpg
│   |   ...
├── label_list.txt
├── test.txt
├── train.txt
└── valid.txt

# label_list.txt 是类别名称列表，文件名必须是 label_list.txt
>>cat label_list.txt
crosswalk
speedlimit
stop
trafficlight

# train.txt 是训练数据集文件列表，每一行是一张图像路径和对应标注文件路径，以空格分开。注意这里的路径是数据集文件夹内的相对路径。
>>cat train.txt
./images/road839.png ./annotations/road839.xml
./images/road363.png ./annotations/road363.xml
...

# valid.txt 是验证数据集文件列表，每一行是一张图像路径和对应标注文件路径，以空格分开。注意这里的路径是数据集文件夹内的相对路径。
>>cat valid.txt
./images/road218.png ./annotations/road218.xml
./images/road681.png ./annotations/road681.xml

也可以使用这个脚本： 

```python
# make-voc-dataset.py
import random
import os
# 生成train.txt和val.txt
random.seed(2020)
xml_dir = './annotations'
img_dir = './images'
path_list = list()
for img in os.listdir(img_dir):
    img_path = os.path.join(img_dir,img)
    xml_path = os.path.join(xml_dir,img.replace('.jpg', '.xml'))
    path_list.append((img_path, xml_path))
random.shuffle(path_list)
ratio = 0.9
train_f = open('./train.txt', 'w')
val_f = open('./valid.txt', 'w')

for i ,content in enumerate(path_list):
    img, xml = content
    text = img + ' ' + xml + '\n'
    if i < len(path_list) * ratio:
        train_f.write(text)
    else:
        val_f.write(text)
train_f.close()
val_f.close()

# 根据自己数据类别生成标签文档
label = ['Rock', 'Paper', 'Scissors']
with open('./label_list.txt', 'w') as f:
    for text in label:
        f.write(text + '\n')

```


```shell
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset# mkdir Rock-Paper-Scissors-VOC
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset# scp -r Rock-Paper-Scissors/* Rock-Paper-Scissors-VOC/
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset# cd Rock-Paper-Scissors-VOC


(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-VOC# ll
total 1524
drwxr-xr-x 4 root root   4096 Sep 12 07:21 ./
drwxr-xr-x 9 root root   4096 Sep 12 07:16 ../
drwxr-xr-x 2 root root 782336 Sep 12 07:21 annotations/
drwxr-xr-x 2 root root 765952 Sep 12 07:21 images/


(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-VOC# vim mkVOC.py 

(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-VOC# python mkVOC.py 
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-VOC# ll
total 2588
drwxr-xr-x 4 root root   4096 Sep 12 07:23 ./
drwxr-xr-x 9 root root   4096 Sep 12 07:16 ../
drwxr-xr-x 2 root root 782336 Sep 12 07:21 annotations/
drwxr-xr-x 2 root root 765952 Sep 12 07:21 images/
-rw-r--r-- 1 root root     20 Sep 12 07:23 label_list.txt
-rw-r--r-- 1 root root    847 Sep 12 07:23 mkVOC.py
-rw-r--r-- 1 root root 973432 Sep 12 07:23 train.txt
-rw-r--r-- 1 root root 107809 Sep 12 07:23 valid.txt


(base) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-COCO# mkdir annotations

```


## 数据集制作-COCO

在 /tools/ 中提供了 x2coco.py 用于将 VOC 数据集转换为COCO数据


```shell
python tools/x2coco.py \
        --dataset_type voc \
        --voc_anno_dir dataset/Rock-Paper-Scissors-VOC/ \
        --voc_anno_list dataset/Rock-Paper-Scissors-VOC/train.txt \
        --voc_label_list dataset/Rock-Paper-Scissors-VOC/label_list.txt \
        --voc_out_name dataset/Rock-Paper-Scissors-COCO/annotations/rpscoco_train.json
```

```
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO# python tools/x2coco.py \
        --dataset_type voc \
        --voc_anno_dir dataset/Rock-Paper-Scissors-VOC/ \
        --voc_anno_list dataset/Rock-Paper-Scissors-VOC/train.txt \
        --voc_label_list dataset/Rock-Paper-Scissors-VOC/label_list.txt \
        --voc_out_name dataset/Rock-Paper-Scissors-COCO/annotations/rpscoco_train.json
Start converting !
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5810/5810 [00:00<00:00, 18739.68it/s]
```

```shell
python tools/x2coco.py \
        --dataset_type voc \
        --voc_anno_dir dataset/Rock-Paper-Scissors-VOC/ \
        --voc_anno_list dataset/Rock-Paper-Scissors-VOC/valid.txt \
        --voc_label_list dataset/Rock-Paper-Scissors-VOC/label_list.txt \
        --voc_out_name dataset/Rock-Paper-Scissors-COCO/annotations/rpscoco_valid.json

```

```shell
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-COCO# ll
total 12
drwxr-xr-x  3 root root 4096 Sep 12 07:45 ./
drwxr-xr-x 10 root root 4096 Sep 12 07:38 ../
drwxr-xr-x  2 root root 4096 Sep 12 08:03 annotations/


(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-COCO# scp -r ../Rock-Paper-Scissors-VOC/images/ ./
(env_paddle) root@143e9ad9746a:/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors-COCO# ll
total 760
drwxr-xr-x  4 root root   4096 Sep 12 08:04 ./
drwxr-xr-x 10 root root   4096 Sep 12 07:38 ../
drwxr-xr-x  2 root root   4096 Sep 12 08:03 annotations/
drwxr-xr-x  2 root root 765952 Sep 12 08:04 images/
```



annotations中的rpscoco_train.json rpscoco_valid.json 文件开头添加infor 和 license 字段： 

```json
    "info": {
        "description": "Rock-Paper-Scissors Dataset",
        "url": "https://your-dataset-url.com",
        "version": "1.0",
        "year": 2025,
        "contributor": "Your Name or Organization",
        "date_created": "2025-09-12"
    },
    "licenses": [
        {
            "url": "https://creativecommons.org/licenses/by/4.0/",
            "id": 1,
            "name": "CC BY 4.0"
        }
    ],  

```

数据集注册： 
参考： PaddleYOLO/configs/datasets/coco_detection.yml
编写： coco_detection_rps.yml

```yml
metric: COCO
num_classes: 3

TrainDataset:
  name: COCODataSet
  image_dir: images
  anno_path: annotations/rpscoco_train.json
  dataset_dir: dataset/Rock-Paper-Scissors-COCO
  data_fields: ['image', 'gt_bbox', 'gt_class', 'is_crowd']

EvalDataset:
  name: COCODataSet
  image_dir: images
  anno_path: annotations/rpscoco_valid.json
  dataset_dir: dataset/Rock-Paper-Scissors-COCO

TestDataset:
  name: ImageFolder
  anno_path: annotations/rpscoco_valid.json # also support txt (like VOC's label_list.txt)
  dataset_dir: dataset/Rock-Paper-Scissors-COCO # if set, anno_path will be 'dataset_dir/anno_path'



```


# 模型训练，基于Rock-Paper-Scissors-COCO数据集

YOLO11-s 预训练模型进行训练

| 网络网络   | 输入尺寸 | 图片数/GPU | 学习率策略 | mAP<sup>val<br>0.5:0.95 | mAP<sup>val<br>0.5 | Params(M) | FLOPs(G) |    下载链接       | 配置文件 |
| :-------: | :----: | :-------: | :------: | :---------------------: | :----------------: |:---------: | :------: |:---------------: |:-----: |
| *YOLO11-n |  640   |     16    |   600e   |  39.4  | 55.3 |  2.6   | 6.5   | [下载链接](https://paddledet.bj.bcebos.com/models/yolo11_n_600e_coco.pdparams) | [配置文件](./yolo11_n_600e_coco.yml) |
| *YOLO11-s |  640   |     16    |   600e   |  46.9  | 63.9 |  9.4   | 21.5  | [下载链接](https://paddledet.bj.bcebos.com/models/yolo11_s_600e_coco.pdparams) | [配置文件](./yolo11_s_600e_coco.yml) |
| *YOLO11-m |  640   |     16    |   600e   |  51.6  | 68.5 |  20.1  | 68.0  | [下载链接](https://paddledet.bj.bcebos.com/models/yolo11_m_600e_coco.pdparams) | [配置文件](./yolo11_m_600e_coco.yml) |
| *YOLO11-l |  640   |     16    |   600e   |  53.3  | 70.1 |  25.3  | 86.9  | [下载链接](https://paddledet.bj.bcebos.com/models/yolo11_l_600e_coco.pdparams) | [配置文件](./yolo11_l_600e_coco.yml) |
| *YOLO11-x |  640   |     16    |   600e   |  54.7  | 71.7 |  56.9  | 194.9 | [下载链接](https://paddledet.bj.bcebos.com/models/yolo11_x_600e_coco.pdparams) | [配置文件](./yolo11_x_600e_coco.yml) |

**注意:**
  - YOLO11模型mAP为部署权重在COCO val2017上的`mAP(IoU=0.5:0.95)`结果，且评估未使用`multi_label`等trick；
  - YOLO11模型训练使用COCO train2017作为训练集，Box AP为在COCO val2017上的`mAP(IoU=0.5:0.95)`结果；
  - YOLO11模型训练过程中默认使用8 GPUs进行混合精度训练，默认lr为0.01为8卡总batch_size的设置，如果**GPU卡数**或者每卡**batch size**发生改动，也不需要改动学习率，但为了保证高精度最好使用**总batch size大于64**的配置去训练；

模型配置文件制作

参考configs/yolo11/yolo11_s_600e_coco.yml , 写一个 yolo11_s_600e_coco_rps.yml


```yml
_BASE_: [
  '../datasets/coco_detection_rps.yml',
  '../runtime.yml',
  '_base_/optimizer_600e.yml',
  '_base_/yolo11_cspdarknet.yml',
  '_base_/yolo11_reader_high_aug.yml',
]

depth_mult: 0.5
width_mult: 0.5

log_iter: 50
snapshot_epoch: 10
weights: output/yolo11_s_600e_coco_rps/model_final


TrainReader:
  batch_size: 16 # default 8 gpus, total bs = 128

TrainDataset:
  refine_bbox: True

```

### 训练
执行以下指令使用混合精度训练YOLO11
```bash
python -m paddle.distributed.launch --gpus 0,1,2,3 tools/train.py -c configs/yolo11/yolo11_s_600e_coco_rps.yml --amp --eval
```

### 评估
执行以下命令在单个GPU上评估RPS COCO数据集
```bash
CUDA_VISIBLE_DEVICES=0 python tools/eval.py -c configs/yolo11/yolo11_s_600e_coco_rps.yml -o weights=output/yolo11_s_600e_coco_rps/best_model.pdparams
```

### 推理
使用以下命令在单张GPU上预测图片，使用`--infer_img`推理单张图片以及使用`--infer_dir`推理文件中的所有图片。
```bash
# 推理单张图片
CUDA_VISIBLE_DEVICES=0 python tools/infer.py -c configs/yolo11/yolo11_s_600e_coco_rps.yml -o weights=output/yolo11_s_600e_coco_rps/best_model.pdparams --infer_img=/workspace/PaddleYOLO/dataset/Rock-Paper-Scissors/images/Photo-on-2-16-22-at-10-19-AM-9_jpg.rf.79e2cf3a561a19b33a74e9ea9477df76.jpg

# 推理文件中的所有图片
CUDA_VISIBLE_DEVICES=0 python tools/infer.py -c configs/yolo11/yolo11_s_600e_coco_rps.yml -o weights=output/yolo11_s_600e_coco_rps/best_model.pdparams --infer_dir=demo
```

![image.png](../assets/01-yolo11-rock-paper-scissors-4.png)

### 导出模型
YOLO11在GPU上推理部署或benchmark测速等需要通过`tools/export_model.py`导出模型。

当你**使用Paddle Inference但不使用TensorRT**时，运行以下的命令导出模型

```bash
python tools/export_model.py -c configs/yolo11/yolo11_n_600e_coco.yml -o weights=output/yolo11_s_600e_coco_rps/best_model.pdparams
```

当你**使用Paddle Inference且使用TensorRT**时，需要指定`-o trt=True`来导出模型。

```bash
python tools/export_model.py -c configs/yolo11/yolo11_n_600e_coco.yml -o weights=output/yolo11_s_600e_coco_rps/best_model.pdparams trt=True
```

如果你想将YOLO11模型导出为**ONNX格式**，参考
[PaddleDetection模型导出为ONNX格式教程](../../deploy/EXPORT_ONNX_MODEL.md)，运行以下命令：

```bash

# 导出推理模型
python tools/export_model.py -c configs/yolo11/yolo11_n_600e_coco.yml --output_dir=output_inference -o weights=output/yolo11_s_600e_coco_rps/best_model.pdparams

# 安装paddle2onnx
pip install paddle2onnx

# 转换成onnx格式
paddle2onnx --model_dir output_inference/yolo11_n_600e_coco --model_filename model.pdmodel --params_filename model.pdiparams --opset_version 11 --save_file yolo11_n_600e_coco.onnx
```